### IMPORT LIBRARIES

In [1]:
import pandas as pd
import numpy as np

import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import (train_test_split,cross_val_score)

from sklearn.preprocessing import LabelEncoder

from sklearn.metrics import (classification_report,roc_auc_score,precision_score,recall_score,f1_score,precision_recall_curve)

from lightgbm import LGBMClassifier

import mlflow
import mlflow.sklearn

### LOAD DATA

In [2]:
df = pd.read_csv("../data/application_train.csv")
bureau = pd.read_csv("../data/bureau.csv")

print(df.shape)
print(bureau.shape)

(307511, 122)
(1716428, 17)


### BUREAU AGGREGATION

In [3]:
bureau_agg = bureau.groupby('SK_ID_CURR').agg({
    'SK_ID_BUREAU': 'count',
    'DAYS_CREDIT': ['min', 'max', 'mean'],
    'CREDIT_DAY_OVERDUE': ['max', 'mean'],
    'AMT_CREDIT_SUM': ['sum', 'mean'],
    'AMT_CREDIT_SUM_DEBT': ['sum', 'mean'],
    'AMT_CREDIT_SUM_OVERDUE': ['mean'],
    'CNT_CREDIT_PROLONG': ['sum']
})

bureau_agg.columns = ['BUREAU_' + '_'.join(col).upper()for col in bureau_agg.columns]
bureau_agg.reset_index(inplace=True)

### MERGE BUREAU

In [4]:
df = df.merge(bureau_agg,on='SK_ID_CURR',how='left')

print("\nMerged Shape:")
print(df.shape)


Merged Shape:
(307511, 134)


### DROP LOW IMPORTANCE COLUMNS

In [5]:
drop_cols = [
    'FLAG_DOCUMENT_18',
    'FLAG_DOCUMENT_19',
    'FLAG_DOCUMENT_20',
    'FLAG_DOCUMENT_21',
    'AMT_REQ_CREDIT_BUREAU_HOUR',
    'AMT_REQ_CREDIT_BUREAU_DAY',
    'AMT_REQ_CREDIT_BUREAU_WEEK',
    'AMT_REQ_CREDIT_BUREAU_MON',
    'AMT_REQ_CREDIT_BUREAU_QRT',
    'AMT_REQ_CREDIT_BUREAU_YEAR',
    'FLAG_DOCUMENT_8',
    'FLAG_DOCUMENT_9',
    'FLAG_DOCUMENT_10',
    'FLAG_DOCUMENT_11',
    'FLAG_DOCUMENT_12',
    'FLAG_DOCUMENT_13',
    'FLAG_DOCUMENT_14',
    'FLAG_DOCUMENT_15',
    'FLAG_DOCUMENT_16',
    'FLAG_DOCUMENT_17',
    'FLAG_DOCUMENT_2',
    'FLAG_DOCUMENT_3',
    'FLAG_DOCUMENT_4',
    'FLAG_DOCUMENT_5',
    'FLAG_DOCUMENT_6',
    'FLAG_DOCUMENT_7',
    'OBS_60_CNT_SOCIAL_CIRCLE',
    'DEF_60_CNT_SOCIAL_CIRCLE',
    'DEF_30_CNT_SOCIAL_CIRCLE',
    'REGION_POPULATION_RELATIVE',
    'FLOORSMAX_MEDI',
    'FLOORSMIN_MEDI',
    'LIVINGAREA_MEDI',
    'NONLIVINGAPARTMENTS_MEDI',
    'NONLIVINGAREA_MEDI',
    'OBS_30_CNT_SOCIAL_CIRCLE',
    'DAYS_LAST_PHONE_CHANGE',
    'DAYS_ID_PUBLISH',
    'CNT_CHILDREN',
    'BASEMENTAREA_MEDI',
    'YEARS_BUILD_MEDI',
    'COMMONAREA_MEDI',
    'ELEVATORS_MEDI',
    'ENTRANCES_MEDI',
    'APARTMENTS_MEDI',
    'FLAG_MOBIL',
    'YEARS_BEGINEXPLUATATION_MEDI',
    'NONLIVINGAREA_MODE',
    'FLOORSMIN_MODE',
    'ELEVATORS_MODE',
    'ENTRANCES_MODE',
    'FLOORSMAX_MODE',
    'FLAG_EMP_PHONE',
    'LIVINGAPARTMENTS_MEDI',
    'LANDAREA_MEDI',
    'NONLIVINGAPARTMENTS_MODE',
    'LIVINGAPARTMENTS_MODE',
    'LANDAREA_MODE',
    'COMMONAREA_MODE',
    'YEARS_BUILD_MODE',
    'YEARS_BEGINEXPLUATATION_MODE',
    'FLAG_WORK_PHONE',
    'FLAG_CONT_MOBILE',
    'FLOORSMIN_AVG',
    'LANDAREA_AVG',
    'LIVINGAPARTMENTS_AVG',
    'LIVINGAREA_AVG',
    'NONLIVINGAPARTMENTS_AVG',
    'NONLIVINGAREA_AVG',
    'APARTMENTS_MODE',
    'BASEMENTAREA_MODE',
    'APARTMENTS_AVG',
    'BASEMENTAREA_AVG',
    'YEARS_BEGINEXPLUATATION_AVG',
    'YEARS_BUILD_AVG',
    'COMMONAREA_AVG',
    'ELEVATORS_AVG',
    'ENTRANCES_AVG',
    'FLOORSMAX_AVG',
    'FLAG_PHONE',
    'REG_REGION_NOT_WORK_REGION',
    'LIVE_REGION_NOT_WORK_REGION',
    'REG_CITY_NOT_LIVE_CITY',
    'REG_CITY_NOT_WORK_CITY',
    'LIVE_CITY_NOT_WORK_CITY',
    'REGION_RATING_CLIENT',
    'REGION_RATING_CLIENT_W_CITY',
    'HOUR_APPR_PROCESS_START',
    'REG_REGION_NOT_LIVE_REGION',
    'NAME_TYPE_SUITE',
    'WALLSMATERIAL_MODE',
    'EMERGENCYSTATE_MODE',
    'WEEKDAY_APPR_PROCESS_START'
]

drop_cols = [col for col in drop_cols if col in df.columns]

df.drop(columns=drop_cols,inplace=True)

print("\nDropped Columns:", len(drop_cols))
print("Shape After Dropping:", df.shape)


Dropped Columns: 93
Shape After Dropping: (307511, 41)


### HANDLE SPECIAL VALUES

In [6]:
df['DAYS_EMPLOYED'] = df['DAYS_EMPLOYED'].replace(365243,np.nan)

### HANDLE MISSING VALUES

In [9]:
for col in df.columns:
    if df[col].dtype == 'object':
        df[col].fillna(df[col].mode()[0],inplace=True)
    else:
        df[col].fillna(df[col].median(),inplace=True)

C:\Users\Fathima Abbas\AppData\Local\Temp\ipykernel_14824\2948150284.py:5: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].fillna(df[col].median(),inplace=True)
C:\Users\Fathima Abbas\AppData\Local\Temp\ipykernel_14824\2948150284.py:5: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a c

### LABEL ENCODING

In [10]:
from category_encoders import TargetEncoder
cat_cols = df.select_dtypes(include=['object']).columns
encoder = TargetEncoder(cols=cat_cols)
df[cat_cols] = encoder.fit_transform(
    df[cat_cols],
    df['TARGET']
)

### FEATURE ENGINEERING

In [11]:
df['AGE'] = abs(df['DAYS_BIRTH']) / 365
df['YEARS_EMPLOYED'] = abs(df['DAYS_EMPLOYED']) / 365
df['EMPLOYED_BIRTH_RATIO'] = (df['DAYS_EMPLOYED'] /(df['DAYS_BIRTH'] + 1))
df['GOODS_INCOME_RATIO'] = (df['AMT_GOODS_PRICE'] /(df['AMT_INCOME_TOTAL'] + 1))
df['EXT_SOURCE_RANGE'] = (df[['EXT_SOURCE_1','EXT_SOURCE_2','EXT_SOURCE_3']].max(axis=1)-df[['EXT_SOURCE_1','EXT_SOURCE_2','EXT_SOURCE_3']].min(axis=1))

# ratios
df['CREDIT_INCOME_RATIO'] = (df['AMT_CREDIT'] /(df['AMT_INCOME_TOTAL'] + 1))
df['ANNUITY_INCOME_RATIO'] = (df['AMT_ANNUITY'] /(df['AMT_INCOME_TOTAL'] + 1))
df['PAYMENT_RATE'] = (df['AMT_ANNUITY'] /(df['AMT_CREDIT'] + 1))
df['GOODS_CREDIT_RATIO'] = (df['AMT_GOODS_PRICE'] /(df['AMT_CREDIT'] + 1))

# ext source features
ext_cols = ['EXT_SOURCE_1','EXT_SOURCE_2','EXT_SOURCE_3']
df['EXT_MEAN'] = df[ext_cols].mean(axis=1)
df['EXT_STD'] = df[ext_cols].std(axis=1)
df['EXT_MIN'] = df[ext_cols].min(axis=1)
df['EXT_MAX'] = df[ext_cols].max(axis=1)
df['EXT_SUM'] = df[ext_cols].sum(axis=1)

# bureau features
if 'BUREAU_AMT_CREDIT_SUM_DEBT_MEAN' in df.columns:
    df['BUREAU_DEBT_RATIO'] = (df['BUREAU_AMT_CREDIT_SUM_DEBT_MEAN'] /(df['BUREAU_AMT_CREDIT_SUM_MEAN'] + 1))

if 'BUREAU_AMT_CREDIT_SUM_OVERDUE_MEAN' in df.columns:
    df['BUREAU_OVERDUE_RATIO'] = (df['BUREAU_AMT_CREDIT_SUM_OVERDUE_MEAN'] /(df['BUREAU_AMT_CREDIT_SUM_MEAN'] + 1))

### LOG TRANSFORMATION

In [12]:
log_cols = ['AMT_INCOME_TOTAL','AMT_CREDIT','AMT_ANNUITY','AMT_GOODS_PRICE']

for col in log_cols:
    df[col] = np.log1p(df[col])

### FEATURE/TARGET

In [13]:
X = df.drop(['TARGET', 'SK_ID_CURR'],axis=1)
y = df['TARGET']

print("\nFinal Shape:")
print(X.shape)


Final Shape:
(307511, 55)


### REMOVE VERY WEAK FEATURES

In [14]:
corr = abs(df.corr()['TARGET'])
remove_cols = corr[corr < 0.005].index
remove_cols = [
    col for col in remove_cols
    if col != 'TARGET'
]
df.drop(
    columns=remove_cols,
    inplace=True
)
print("\nRemoved Columns:")
print(len(remove_cols))



Removed Columns:
10


### REMOVE HIGH CORRELATED FEATURES